# API QoS Estimation

In [1]:
# First we import the requested modules
import json

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"
pd.set_option("display.precision", 3)

import multiprocessing
from datetime import timedelta

from pandarallel import pandarallel

num_cores = multiprocessing.cpu_count()
pandarallel.initialize()

INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [2]:
# Available colors
colors = [
    "#1f77b4",  # muted blue
    "#ff7f0e",  # safety orange
    "#2ca02c",  # cooked asparagus green
    "#d62728",  # brick red
    "#9467bd",  # muted purple
    "#8c564b",  # chestnut brown
    "#e377c2",  # raspberry yogurt pink
    "#7f7f7f",  # middle gray
    "#bcbd22",  # curry yellow-green
    "#17becf",  # blue-teal
]

In [3]:
# FUNCTIONS
def str_to_int(string):
    final_val = 0
    for c in string:
        val = ord(c)
        final_val += val
    return final_val

In [4]:
f = "../Data/"

In [5]:
# Load line_stops_dict
with open(f + "Static/lines_dict.json") as file:
    lines_dict = json.load(file)

## Last week's data

In [6]:
# Read week data — live from SQLite telemetry engine
import os
import sys

nb_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(nb_dir, "..", ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from core import db

week_df = db.get_latest_burst_df("Madrid")
week_df["line"] = week_df["line"].astype(str)
week_df["datetime"] = pd.to_datetime(week_df["datetime"], format="ISO8601")

In [7]:
def add_direction(row):
    # Assign destination values
    dest2, dest1 = lines_dict[str(row.line)]["destinations"]

    direction = 1 if row.destination == dest1 else 2
    return direction


# Add direction field to df
week_df["direction"] = week_df.apply(add_direction, axis=1)

In [8]:
week_df.head()

,line,bus,vehicleId,destination,stop,estimateArrive,DistanceBus,lat,lon,datetime,direction
0,82,4713,,PITIS,1730,88,560.0,0.0,0.0,2021-03-17 16:46:04.125767,1
1,82,4710,,PITIS,1730,472,2813.0,0.0,0.0,2021-03-17 16:46:04.125767,1
2,82,4713,,PITIS,1730,88,560.0,0.0,0.0,2021-03-17 16:46:04.125767,1
3,82,4710,,PITIS,1730,472,2813.0,0.0,0.0,2021-03-17 16:46:04.125767,1


In [9]:
week_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   line            4 non-null      str           
 1   bus             4 non-null      int64         
 2   vehicleId       4 non-null      str           
 3   destination     4 non-null      str           
 4   stop            4 non-null      str           
 5   estimateArrive  4 non-null      int64         
 6   DistanceBus     4 non-null      float64       
 7   lat             4 non-null      float64       
 8   lon             4 non-null      float64       
 9   datetime        4 non-null      datetime64[us]
 10  direction       4 non-null      int64         
dtypes: datetime64[us](1), float64(3), int64(3), str(4)
memory usage: 484.0 bytes


# Analysis of temporal series belonging to a bus
Analyze all the data corresponding to the different trips of a bus to a stop.
We pay attention to the last TH values of the series.

In [10]:
# Number of last ocurrences which form the series we are going to analyze for QoS
TH = 30

In [11]:
th_df = week_df.sort_values(by=["bus", "stop", "datetime"], ascending=True)
th_df = th_df.drop_duplicates(["bus", "stop", "datetime"], keep="last")
th_df = th_df[th_df.datetime > th_df.datetime.max() - timedelta(seconds=900)]
th_df.tail(5)

,line,bus,vehicleId,destination,stop,estimateArrive,DistanceBus,lat,lon,datetime,direction
3,82,4710,,PITIS,1730,472,2813.0,0.0,0.0,2021-03-17 16:46:04.125767,1
2,82,4713,,PITIS,1730,88,560.0,0.0,0.0,2021-03-17 16:46:04.125767,1


In [12]:
th_df[th_df.line == 132]

,line,bus,vehicleId,destination,stop,estimateArrive,DistanceBus,lat,lon,datetime,direction


In [13]:
def build_time_series_graph(th_df, TH, bus_id):

    graph = go.Figure()

    # TH_DF
    series_df = th_df[th_df.datetime > th_df.datetime.max() - timedelta(seconds=TH * 30)]

    # Loc Bus Appearances
    series_df = series_df[series_df.bus == bus_id]

    if series_df.shape[0] < 1:
        return graph
    line = series_df.line.iloc[0]
    direction = series_df.direction.iloc[0]
    stops_list = lines_dict[str(line)][str(direction)]["stops"]

    # Set title and layout
    graph.update_layout(
        title=f"<b>Bus {bus_id} : ETA Time Series</b> - Line: {line}",
        legend_title="<b>Destination Stop</b>",
        yaxis={"title": "ETA in Seconds", "nticks": 10, "zerolinecolor": "darkgrey"},
        margin={"r": 0, "l": 0, "t": 40, "b": 0},
        hovermode="closest",
    )

    # Locate unique stops
    unique_stops = series_df.stop.unique().tolist()
    for stop in stops_list:
        if stop not in unique_stops:
            continue
        else:
            stop_index = stops_list.index(stop)

        stop_df = series_df[series_df.stop == stop]

        # Build stop trace
        graph.add_trace(
            go.Scatter(
                name="[" + str(stop_index) + "] " + str(stop),
                x=stop_df.datetime,
                y=stop_df.estimateArrive,
                mode="lines+markers",
                line={"width": 3, "color": colors[(str_to_int(stop)) % len(colors)]},
                text=[
                    "<b>Bus : "
                    + str(bus_id)
                    + "</b> <br>"
                    + "Stop["
                    + str(stop_index)
                    + "]: "
                    + str(stop)
                    + "<br>"
                    + "Time : "
                    + row.datetime.strftime("%H:%M:%S")
                    + "<br>"
                    + "ETA : "
                    + str(row.estimateArrive)
                    for row in stop_df.itertuples()
                ],
                hoverinfo="text",
            )
        )

    return graph

In [14]:
bus_id = "4707"
build_time_series_graph(th_df, TH, bus_id).show()